# Underperforming Backtest Cleanup v1

Goal: compare current replay/backtest surfaces against `soxl_backtest_policy_reimplementation_v1`, keep executable paths that are not worse on the 2026 YTD common window, protect production/runtime-component paths, and prune only duplicated underperforming executable artifacts while retaining their ideas.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'AGENTS.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import audit_underperforming_backtests as audit

CARD_DIR = ROOT / 'runtime/production/card/underperforming_backtest_cleanup_v1'
SUMMARY_PATH = CARD_DIR / 'summary.json'
DECISIONS_PATH = CARD_DIR / 'cleanup_decisions.csv'
DELETE_MANIFEST_PATH = CARD_DIR / 'delete_manifest.csv'
IDEAS_PATH = CARD_DIR / 'retained_ideas.md'

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
decisions = pd.read_csv(DECISIONS_PATH)
delete_manifest = pd.read_csv(DELETE_MANIFEST_PATH)
summary

{'generated_at': '2026-06-12T07:58:33.964968+00:00',
 'primary_profile': 'soxl_backtest_policy_reimplementation_v1',
 'primary_return_pct': 132.07797795135497,
 'comparison_window': 'ytd_2026_common',
 'decision_counts': {'keep_executable': 11,
  'keep_primary': 1,
  'protect_runtime_component': 1,
  'archive_idea_only': 1,
  'protect_production_runtime': 1},
 'cleanup_action_counts': {'none': 14,
  'delete_profile_and_actor_checkpoints': 1},
 'applied': {'deleted': ['runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/profile.yaml',
   'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/soxl_balanced_rule_residual_v22_continued_retention_v1_actor.pt',
   'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/soxl_balanced_rule_residual_v22_continued_retention_v1_checkpoint_iter0_actor.pt',
   'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/soxl_balanced_rule_residual_v22_continued_retention_v1_checkpoint_iter10_ac

## Decision Table

In [2]:
display(
    decisions[
        [
            'profile',
            'status',
            'ytd_return_pct',
            'return_delta_vs_primary_pct',
            'decision',
            'cleanup_action',
        ]
    ]
)
display(delete_manifest)

,profile,status,ytd_return_pct,return_delta_vs_primary_pct,decision,cleanup_action
0,soxl_backtest_policy_reimplementation_v1,primary_reimplementation,132.077978,0.000000,keep_primary,none
1,smoke,canonical,224.361539,92.283561,keep_executable,none
2,balanced_macro_v1,candidate,196.749008,64.671031,keep_executable,none
3,state_conditioned_drawdown_rebound_reward_trai...,ablation,196.505054,64.427076,keep_executable,none
4,mode_conditioned_rebound_reward_train_v2,ablation,177.573569,45.495591,keep_executable,none
5,balanced_rule_residual_v19_reg_v22_retest_v1,ablation,175.012377,42.934399,keep_executable,none
6,balanced_rule_residual_v19,archived,173.299227,41.221249,keep_executable,none
7,drawdown_rebound_reward_train_v1,ablation,172.832548,40.754570,keep_executable,none
8,balanced_rule_residual_v19_reg_light,candidate,164.253638,32.175660,keep_executable,none
9,balanced,archived,159.205721,27.127743,keep_executable,none


,profile,decision,cleanup_action,path
0,balanced_rule_residual_v22_continued_retention_v1,archive_idea_only,delete_profile_and_actor_checkpoints,runtime/model/card/balanced_rule_residual_v22_...
1,balanced_rule_residual_v22_continued_retention_v1,archive_idea_only,delete_profile_and_actor_checkpoints,runtime/model/card/balanced_rule_residual_v22_...
2,balanced_rule_residual_v22_continued_retention_v1,archive_idea_only,delete_profile_and_actor_checkpoints,runtime/model/card/balanced_rule_residual_v22_...
3,balanced_rule_residual_v22_continued_retention_v1,archive_idea_only,delete_profile_and_actor_checkpoints,runtime/model/card/balanced_rule_residual_v22_...
4,balanced_rule_residual_v22_continued_retention_v1,archive_idea_only,delete_profile_and_actor_checkpoints,runtime/model/card/balanced_rule_residual_v22_...


## Cleanup Verification

In [3]:
primary = decisions.loc[decisions['profile'].eq(audit.PRIMARY_PROFILE)].iloc[0]
v22 = decisions.loc[decisions['profile'].eq('balanced_rule_residual_v22')].iloc[0]
v22_continued = decisions.loc[decisions['profile'].eq(audit.V22_CONTINUED_PROFILE)].iloc[0]
composer = decisions.loc[decisions['profile'].eq('v19reg_v22_budget_usage_bull_guard_v1')].iloc[0]

assert primary['decision'] == 'keep_primary'
assert abs(float(primary['ytd_return_pct']) - 132.07797795135497) < 1e-9
assert v22['decision'] == 'protect_runtime_component'
assert v22_continued['status'] == 'dropped'
assert v22_continued['decision'] == 'archive_idea_only'
assert v22_continued['cleanup_action'] == 'delete_profile_and_actor_checkpoints'
assert composer['decision'] == 'protect_production_runtime'

deleted_paths = [Path(path) for path in summary['applied']['deleted']]
still_present = [str(path) for path in deleted_paths if (ROOT / path).exists()]
assert not still_present, still_present
assert summary['applied']['deleted_count'] == len(deleted_paths) == 5
assert summary['applied']['freed_bytes'] == 747810

retained_paths = [
    ROOT / 'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/notes.md',
    ROOT / 'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/soxl_balanced_rule_residual_v22_continued_retention_v1_summary.json',
    ROOT / 'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/v22_continued_retention_review.md',
    IDEAS_PATH,
]
missing_retained = [str(path.relative_to(ROOT)) for path in retained_paths if not path.exists()]
assert not missing_retained, missing_retained

{
    'primary_return_pct': float(primary['ytd_return_pct']),
    'v22_delta_vs_primary_pct': float(v22['return_delta_vs_primary_pct']),
    'deleted_count': summary['applied']['deleted_count'],
    'freed_bytes': summary['applied']['freed_bytes'],
    'retained_idea_files': [str(path.relative_to(ROOT)) for path in retained_paths],
}

{'primary_return_pct': 132.07797795135497,
 'v22_delta_vs_primary_pct': -15.195334263284607,
 'deleted_count': 5,
 'freed_bytes': 747810,
 'retained_idea_files': ['runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/notes.md',
  'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/soxl_balanced_rule_residual_v22_continued_retention_v1_summary.json',
  'runtime/model/card/balanced_rule_residual_v22_continued_retention_v1/v22_continued_retention_review.md',
  'runtime/production/card/underperforming_backtest_cleanup_v1/retained_ideas.md']}

## Conclusion

- `soxl_backtest_policy_reimplementation_v1` remains the primary executable rule-policy benchmark: `132.077978%` on the 2026 YTD common window, with trade timing already validated against the legacy non-qlib `soxl_backtest` path.
- 11 replay profiles beat that YTD threshold and are kept executable.
- `balanced_rule_residual_v22` trails the primary benchmark by `15.195334pp` on this narrow YTD replay, but is protected as a canonical runtime component/fast leg rather than deleted.
- `v19reg_v22_budget_usage_bull_guard_v1` trails the primary benchmark by `128.827626pp` on this narrow YTD replay, but is protected as the current production scheduler composer/risk-control path.
- The only executable cleanup applied in this pass is `balanced_rule_residual_v22_continued_retention_v1`: deleted `profile.yaml` plus four actor/checkpoint files, freeing `747810` bytes. Its notes, review, and summary remain as the idea record.